# Distress Gesture Detection — Training Pipeline

### Before running: Enable GPU + Internet + Add 6 Datasets
**Run all cells top to bottom.**

## Cell 1 — Verify GPU + Clone Repo

In [ ]:
import subprocess, os, sys
from pathlib import Path

gpu = subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
print(f'GPU: {gpu}')

print('\nAttached datasets:')
for d in sorted(Path('/kaggle/input').iterdir()):
    print(f'  /kaggle/input/{d.name}')

REPO_URL = 'https://github.com/shrishri12062000/distress.git'
REPO_DIR = '/kaggle/working/distress-gesture-detection'

if Path(REPO_DIR).exists():
    print('\nRepo exists — pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print('\nCloning repo...')
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        print('ERROR: Clone failed. Check Internet is ON in Settings.')

sys.path.insert(0, REPO_DIR)
print(f'\nRepo ready at: {REPO_DIR}')

## Cell 2 — Install Dependencies

In [ ]:
import subprocess

print('Step 1/3 — Installing core packages...')
subprocess.run('pip install -q datasets ultralytics onnx onnxruntime huggingface-hub', shell=True)
print('  core packages done')

print('Step 2/3 — Installing mediapipe (latest for NumPy 2.x)...')
ret = subprocess.run('pip install -q --upgrade mediapipe', shell=True)
print('  mediapipe done')

print('Step 3/3 — Verifying imports...')
import importlib, numpy as np, torch
print(f'  numpy:  {np.__version__}')
print(f'  torch:  {torch.__version__}  CUDA: {torch.cuda.is_available()}')

# Test cv2 separately — may fail if numpy conflict persists
try:
    import cv2
    print(f'  opencv: {cv2.__version__}')
except Exception as e:
    print(f'  opencv: FAILED ({e})')
    print('  Fixing: reinstalling opencv-python-headless...')
    subprocess.run('pip install -q --upgrade opencv-python-headless', shell=True)
    import cv2
    print(f'  opencv: {cv2.__version__} (fixed)')

# Test mediapipe separately
try:
    import mediapipe
    print(f'  mediapipe: {mediapipe.__version__}')
    MEDIAPIPE_OK = True
except Exception as e:
    print(f'  mediapipe: FAILED ({e})')
    print('  Video datasets (URFD, Le2i) will be SKIPPED — NTU + OmniFall will be used')
    MEDIAPIPE_OK = False

try:
    import ultralytics, onnx, onnxruntime
    print(f'  ultralytics: {ultralytics.__version__}')
    print(f'  onnx: {onnx.__version__}')
except Exception as e:
    print(f'  ERROR: {e}')

print(f'\nAll done. mediapipe_ok={MEDIAPIPE_OK}')

In [ ]:
## Cell 2b — Inspect Dataset Layouts (run once to verify paths)
import os
from pathlib import Path

KAGGLE_ROOT = Path('/kaggle/input/datasets')

def show_tree(path, prefix='', max_depth=3, max_files=5, _depth=0):
    p = Path(path)
    if not p.exists():
        print(f'{prefix}[NOT FOUND] {path}')
        return
    entries = sorted(p.iterdir())
    dirs  = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]
    for d in dirs[:max_files]:
        print(f'{prefix}  [{d.name}/]')
        if _depth < max_depth - 1:
            show_tree(d, prefix + '    ', max_depth, max_files, _depth + 1)
    if len(dirs) > max_files:
        print(f'{prefix}  ... ({len(dirs) - max_files} more dirs)')
    for f in files[:max_files]:
        size = f.stat().st_size
        print(f'{prefix}  {f.name}  ({size:,} bytes)')
    if len(files) > max_files:
        print(f'{prefix}  ... ({len(files) - max_files} more files)')

datasets = {
    'NTU':   'hungkhoi/skeleton-data-of-ntu-rgbd-60-dataset',
    'URFD':  'shahliza27/ur-fall-detection-dataset',
    'Le2i':  'tuyenldvn/falldataset-imvia',
    'Knife1': 'simuletic/cctv-weapon-dataset',
    'Knife2': 'simuletic/cctv-atm-robbery-detection-dataset-gun-and-knife',
    'Knife3': 'simuletic/surveillance-vlm-weapon-and-knife-detection-dataset',
}

for name, rel_path in datasets.items():
    full = KAGGLE_ROOT / rel_path
    print(f'\n── {name} ──────────────────────────────────')
    print(f'   {full}')
    show_tree(full, max_depth=3, max_files=6)


## Cell 3 — Data Preparation
⏱ **30–90 minutes.** Do not close the tab.

In [ ]:
os.chdir(REPO_DIR)
%run kaggle/prepare_data.py

## Cell 4 — Validate Data

In [ ]:
%run kaggle/validate_data.py

## Cell 5 — Train ST-GCN
⏱ **2–4 hours on GPU.**

In [ ]:
%run kaggle/train_stgcn.py

## Cell 6 — Train YOLOv8n Knife Detector
⏱ **30–60 minutes.**

In [ ]:
%run kaggle/train_yolo.py

## Cell 7 — Export Models to ONNX

In [ ]:
%run kaggle/export_models.py

## Cell 8 — Verify + Download Models

In [ ]:
import numpy as np
import onnxruntime as ort
from pathlib import Path

MODELS = Path('/kaggle/working')

stgcn_path = MODELS / 'stgcn.onnx'
if stgcn_path.exists():
    sess  = ort.InferenceSession(str(stgcn_path), providers=['CPUExecutionProvider'])
    dummy = np.zeros((1, 3, 30, 17), dtype=np.float32)
    out   = sess.run(None, {sess.get_inputs()[0].name: dummy})[0][0]
    probs = np.exp(out) / np.exp(out).sum()
    print('ST-GCN output:')
    for name, p in zip(['normal','help_signal','collapse_falling','fall_down'], probs):
        print(f'  {name:<20}: {p:.4f}')
    print('  ST-GCN ONNX working')
else:
    print('ST-GCN ONNX not found')

yolo_path = MODELS / 'yolo_knife.onnx'
if yolo_path.exists():
    sess  = ort.InferenceSession(str(yolo_path), providers=['CPUExecutionProvider'])
    dummy = np.zeros((1, 3, 640, 640), dtype=np.float32)
    out   = sess.run(None, {sess.get_inputs()[0].name: dummy})
    print(f'YOLOv8 output shape: {out[0].shape} — working')
else:
    print('YOLOv8 ONNX not found')

print('\n' + '='*50)
print('DOWNLOAD: Right panel → Output → download:')
print('  stgcn.onnx  and  yolo_knife.onnx')
print('Place in: distress-gesture-detection/models/')
print('='*50)